In [1]:
import os
os.environ.setdefault("USER_AGENT", "AI Agents and Agentic Workflows educational RAG notebook")

from langchain_community.document_loaders import WikipediaLoader, WebBaseLoader, Docx2txtLoader, PyPDFLoader, TextLoader

from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama

## Setting up vector database and embeddings

In [2]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
embeddings_model = None  # Use Chroma's local all-MiniLM-L6-v2 embeddings
vector_db = Chroma("tourist_info", embeddings_model)

> **Chroma defaults used here (verified with the installed Chroma 1.3.0):** For a newly created local/single-node collection, `embeddings_model = None` lets Chroma attach its built-in `DefaultEmbeddingFunction`, which uses ONNX Runtime with `all-MiniLM-L6-v2`. The main dense-vector index is **HNSW** (approximate nearest-neighbor search), not IVF or PQ. Its default `space` is **`l2`**, which Chroma defines as squared Euclidean distance: $\sum_i (A_i-B_i)^2$. Therefore, smaller returned distances mean closer matches; this is not cosine similarity or dot product. Newly added vectors first enter a small brute-force buffer (default batch size: 100) before being merged into HNSW. These settings are established when the collection is created.

> Sources: [Chroma index configuration](https://docs.trychroma.com/docs/collections/configure) and [Chroma collection defaults](https://cookbook.chromadb.dev/core/collections/).

In [3]:
try:
    wikipedia_loader = WikipediaLoader(query="Paestum")
    wikipedia_chunks = text_splitter.split_documents(wikipedia_loader.load())
    vector_db.add_documents(wikipedia_chunks)
except Exception as error:
    print(f"Wikipedia API failed ({type(error).__name__}: {error}). Loading the Paestum page directly.")
    wikipedia_loader = WebBaseLoader(
        "https://en.wikipedia.org/wiki/Paestum",
        header_template={"User-Agent": "AI Agents and Agentic Workflows educational RAG notebook"}
    )
    wikipedia_chunks = text_splitter.split_documents(wikipedia_loader.load())
    vector_db.add_documents(wikipedia_chunks)

Wikipedia API failed (JSONDecodeError: Expecting value: line 1 column 1 (char 0)). Loading the Paestum page directly.


In [4]:
word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
word_chunks = text_splitter.split_documents(word_loader.load())
vector_db.add_documents(word_chunks)

['e399fd27-432d-4a6c-ab6e-9cf932a55f08',
 '6367ba68-0fad-4d74-a2e0-2c0098380186',
 'bc9e411f-bbe7-4f78-9c6c-7ff10de90d55',
 '3158b339-f85f-4d92-bd26-100ccbdd02a4',
 'f03f302b-8cc9-4876-a02a-355a8251733e',
 'dc24d48d-5455-4b70-a309-921a14491e6b',
 '849eb9b8-fbbc-46e3-995d-b3e34710453f',
 '304653e0-1722-4765-89c4-202ee182d2da']

In [5]:
pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
pdf_chunks = text_splitter.split_documents(pdf_loader.load())
vector_db.add_documents(pdf_chunks)

['fb9a9d46-69b0-4e39-99df-2630a14d8a6d',
 '2f6b21cc-2e2f-4b72-8afd-590425a50403',
 '6e9d7b1b-7cfe-41f0-a719-dd20493e5d1d',
 'ec937c9f-439c-4506-8e72-e138cf9d40ce',
 '23a79261-d8e4-4096-9f9a-ebf2e28f3d20',
 '462cc4ef-537d-43d6-8ac4-f37f0af3ba72',
 '7b23f7dc-7cbd-4e16-96ab-14beecd63402',
 '073d6fc7-852c-4c0f-b506-bfd638aa6f0a',
 '09b2f503-5a6e-4eef-b4e3-37c6a4dbf229',
 'd7fa3157-d765-48aa-aa6f-0e737756073e',
 'a2f38e07-ea77-489c-9927-d800160067fc',
 'e8768242-d18f-43cf-91a1-f47895c9e9b7',
 'a91ca198-af04-42bc-8e73-faf7e3d07bb7',
 '5f344bb2-66e9-479e-bbef-cc02b28bad6d',
 '80d81de6-6e1e-49f5-8fb4-7c58ed51652d',
 'd31ab277-a4cd-4de8-bcd2-a6fef4e077a3',
 '530fc4c6-8a41-45c4-8947-d00fc5435207',
 '120a595e-5c37-43dd-8721-2c017b9439f3',
 'd58d475a-82d1-4d5b-aa6f-0884c567a596',
 '5e48b1d7-45e7-4a86-9cdd-39804023ac9a',
 '2630bdc7-be65-4e7d-9148-e3c3a5b7fa14',
 'bc9f2727-d5a8-483d-a446-27b440752efb',
 'c0f4c74d-6c9b-4a24-84ef-2e3990b30990',
 'eefab6aa-ff1e-4669-8118-d8f6b6a01e85',
 'e5426b09-53ad-

In [6]:
txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
txt_chunks = text_splitter.split_documents(txt_loader.load())
vector_db.add_documents(txt_chunks)

['22ddd837-183e-4b53-a0f9-bd584eeb558c']

## Removing duplication

In [7]:
def split_and_import(loader):
     chunks = text_splitter.split_documents(loader.load())
     vector_db.add_documents(chunks)
     print(f"Ingested chunks created by {loader}")

In [8]:
try:
    wikipedia_loader = WikipediaLoader(query="Paestum")
    split_and_import(wikipedia_loader)
except Exception as error:
    print(f"Wikipedia API failed ({type(error).__name__}: {error}). Loading the Paestum page directly.")
    wikipedia_loader = WebBaseLoader(
        "https://en.wikipedia.org/wiki/Paestum",
        header_template={"User-Agent": "AI Agents and Agentic Workflows educational RAG notebook"}
    )
    split_and_import(wikipedia_loader)

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
split_and_import(word_loader)

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
split_and_import(pdf_loader)

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
split_and_import(txt_loader)

Wikipedia API failed (JSONDecodeError: Expecting value: line 1 column 1 (char 0)). Loading the Paestum page directly.
Ingested chunks created by <langchain_community.document_loaders.web_base.WebBaseLoader object at 0x0000025504305810>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x00000255012F6990>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x00000255043056D0>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x0000025504306FD0>


## Ingesting Multiple Documents from a Folder (two techniques)

### 1) Iterating over all files in a folder

In [9]:
loader_classes = {
    'docx': Docx2txtLoader,
    'pdf': PyPDFLoader,
    'txt': TextLoader
}

In [10]:
import os

def get_loader(filename):
    _, file_extension = os.path.splitext(filename) #A Extract the file extension
    file_extension = file_extension.lstrip('.') #B Remove the leading dot from the extension

    loader_class = loader_classes.get(
        file_extension) #C Get the loader class from the dictionary

    if loader_class:
        return loader_class(filename) #D Instantiate and return the correct loader
    else:
        raise ValueError(f"No loader available for file extension '{file_extension}'")

### Ingesting the files from the folder (Exercise solution)

In [11]:
folder_path = "CilentoTouristInfo" #A Path to the folder containing the documents

for filename in os.listdir(folder_path): #B iterate over the files in the path
    file_path = os.path.join(folder_path, filename) #C Construct the full path to the file

    if os.path.isfile(file_path): #D Check if it is a file (not a directory)
        try:
            loader = get_loader(file_path) #E Instantiate the correct loader for the file
            print(f"Loader for {filename}: {loader}")
            split_and_import(loader) #F Split and ingest
        except ValueError as e:
            print(e)

Loader for Acciaroli.pdf: <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x0000025504307390>
Ingested chunks created by <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x0000025504307390>
Loader for Cape Palinuro.txt: <langchain_community.document_loaders.text.TextLoader object at 0x0000025504307750>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x0000025504307750>
Loader for Casalvelino.txt: <langchain_community.document_loaders.text.TextLoader object at 0x00000255042EEC40>
Ingested chunks created by <langchain_community.document_loaders.text.TextLoader object at 0x00000255042EEC40>
Loader for Cilentan coast.docx: <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x0000025504307750>
Ingested chunks created by <langchain_community.document_loaders.word_document.Docx2txtLoader object at 0x0000025504307750>
Loader for Cilento Coast Map and Travel Guide.docx: <langchain_community.docum

### 2) Ingesting all files with with DirectoryLoader

In [12]:
# ONLY RUN THIS IF YOU HAVE SUCCESFULLY INSTALLED unstructured or langchain-unstructured
# THE INSTALLATION IS OPERATIVE SYSTEM SPECIFIC
# follow LangChain instructions at https://python.langchain.com/v0.2/docs/integrations/providers/unstructured/ or
# Unstructured instructions at https://docs.unstructured.io/welcome#quickstart-unstructured-open-source-library
folder_path = "CilentoTouristInfo"
pattern = "**/*.{docx,pdf,txt}" #A Pattern to match .docx, .pdf, and .txt files

directory_loader = DirectoryLoader(folder_path, pattern) #B Initialize the DirectoryLoader with the folder path and pattern
split_and_import(directory_loader)

NameError: name 'DirectoryLoader' is not defined

## Querying the vector store directly

In [13]:
query = "Where was Poseidonia and who renamed it to Paestum?"
results = vector_db.similarity_search(query, 4) # four clostest results
print(results)

[Document(id='6698d91a-1eee-4ef2-b78b-f9de9ebc2da6', metadata={'language': 'en', 'title': 'Paestum - Wikipedia', 'source': 'https://en.wikipedia.org/wiki/Paestum'}, page_content='The Greek settlers who founded the city originally named it Poseidonia (Ancient Greek: Ποσειδωνία). It was eventually conquered by the local Lucanians and later the Romans. The Lucanians renamed it to Paistos and the Romans gave the city its current name.[5]\nAncient ruins and features[edit]\nAerial view of Paestum, looking north; two Hera Temples in foreground, Athena Temple in background.'), Document(id='a31fdd83-6eea-4734-8e14-8358849cd4e3', metadata={'source': 'https://en.wikipedia.org/wiki/Paestum', 'language': 'en', 'title': 'Paestum - Wikipedia'}, page_content='The Greek settlers who founded the city originally named it Poseidonia (Ancient Greek: Ποσειδωνία). It was eventually conquered by the local Lucanians and later the Romans. The Lucanians renamed it to Paistos and the Romans gave the city its curr

In [14]:
len(results)

4

## Asking a question through a RAG chain

In [15]:
from langchain_core.prompts import PromptTemplate

rag_prompt_template = """Use the following pieces of context
to answer the question at the end.
If you don't know the answer, just say that you don't know,
don't try to make up an answer.
Use three sentences maximum and keep the
answer as concise as possible.
{context}
Question: {question}
Helpful Answer:"""

rag_prompt = PromptTemplate.from_template(rag_prompt_template)

In [16]:
retriever = vector_db.as_retriever()

In [17]:
from langchain_core.runnables import RunnablePassthrough
question_feeder = RunnablePassthrough()

In [18]:
chatbot = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=65536,
    temperature=0,
    reasoning=False
)

In [19]:
# set up RAG chain

rag_chain = {"context": retriever,
             "question": question_feeder} | rag_prompt | chatbot

In [20]:
def execute_chain(chain, question):
    answer = chain.invoke(question)
    return answer

In [21]:
question = """Where was Poseidonia and who renamed
it to Paestum. Also tell me the source."""
answer = execute_chain(rag_chain, question)
print(answer.content)

Poseidonia is the original name given to the city by Greek settlers, which was later conquered by the Lucanians and Romans. The Romans gave the city its current name of Paestum. The source for this information is Wikipedia.


In [22]:
print(answer)

content='Poseidonia is the original name given to the city by Greek settlers, which was later conquered by the Lucanians and Romans. The Romans gave the city its current name of Paestum. The source for this information is Wikipedia.' additional_kwargs={} response_metadata={'model': 'gemma4:12b-it-q8_0', 'created_at': '2026-08-07T04:03:59.633676Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11018274700, 'load_duration': 5207654600, 'prompt_eval_count': 690, 'prompt_eval_duration': 4711705000, 'eval_count': 48, 'eval_duration': 1091742000, 'logprobs': None, 'model_name': 'gemma4:12b-it-q8_0', 'model_provider': 'ollama'} id='lc_run--019fda64-3705-7810-a901-a943804fb623-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 690, 'output_tokens': 48, 'total_tokens': 738}


In [23]:
question = """And then, what they do?
Tell me only if you know.
Also tell me the source"""
answer = execute_chain(rag_chain, question)
print(answer.content)

I do not know what "they" do based on the provided context. The documents mention accessibility features like ramps and elevators at an archaeological park, but do not describe specific actions performed by people.


## Chatbot memory of message history

In [24]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant, world-class expert in Roman and Greek history, especially in towns located in southern Italy. Provide interesting insights on local history and recommend places to visit with knowledgeable and engaging answers. Answer all questions to the best of your ability, but only use what has been provided in the context. If you don't know, just say you don't know. Use three sentences maximum and keep the answer as concise as possible."),
        ("placeholder", "{chat_history_messages}"),
        ("assistant", "{retrieved_context}"),
        ("human", "{question}"),
    ]
)

retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()
chatbot = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=65536,
    temperature=0,
    reasoning=False
)
chat_history_memory = ChatMessageHistory()

def get_messages(x):
    return chat_history_memory.messages

rag_chain = {
    "retrieved_context": retriever,
    "question": question_feeder,
    "chat_history_messages": RunnableLambda(get_messages)
} | rag_prompt | chatbot

def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'Full chat message history: {chat_history_memory.messages}\n\n')
    return answer

In [25]:
question = """Where was Poseidonia and who renamed
it to Paestum? Also tell me the source."""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed\nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was founded by Greek settlers in the location now known as Paestum. The Lucanians renamed it to Paistos, and the Romans eventually gave the city its current name of Paestum. This information is sourced from Wikipedia.', additional_kwargs={}, response_metadata={'model': 'gemma4:12b-it-q8_0', 'created_at': '2026-08-07T04:04:13.7893466Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1934633700, 'load_duration': 316562100, 'prompt_eval_count': 747, 'prompt_eval_duration': 338552000, 'eval_count': 50, 'eval_duration': 1153887000, 'logprobs': None, 'model_name': 'gemma4:12b-it-q8_0', 'model_provider': 'ollama'}, id='lc_run--019fda64-91cd-7c12-a923-c5e9e15fd60f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 747, 'output_tokens': 50, 'total_tokens': 797})]

In [26]:
question = """And then what did they do?
Also tell me the source"""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed\nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Poseidonia was founded by Greek settlers in the location now known as Paestum. The Lucanians renamed it to Paistos, and the Romans eventually gave the city its current name of Paestum. This information is sourced from Wikipedia.', additional_kwargs={}, response_metadata={'model': 'gemma4:12b-it-q8_0', 'created_at': '2026-08-07T04:04:13.7893466Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1934633700, 'load_duration': 316562100, 'prompt_eval_count': 747, 'prompt_eval_duration': 338552000, 'eval_count': 50, 'eval_duration': 1153887000, 'logprobs': None, 'model_name': 'gemma4:12b-it-q8_0', 'model_provider': 'ollama'}, id='lc_run--019fda64-91cd-7c12-a923-c5e9e15fd60f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 747, 'output_tokens': 50, 'total_tokens': 797}),

## Tracing with LangSmith

Stop the notebook and open a new operative system shell (for example Windows command shell).

Configure the relevant environment variables in the OS shell, the rerun the previous Jupyter cells:
```
(env_ch07) C:\...\ch07>set LANGSMITH_TRACING=true
(env_ch07) C:\...\ch07>set LANGSMITH_ENDPOINT=https://api.smith.langchain.com
(env_ch07) C:\...\ch07>set LANGSMITH_PROJECT=Q & A chatbot
(env_ch07) C:\...\ch07>set LANGSMITH_API_KEY=<YOUR_LANGSMITH_API_KEY>
```
Then Restart the Jupyter notebook:

```(env_ch07) C:\...\ch07>jupyter notebook 07-QA_across_documents.ipynb```

Finally re-execute the whole Jupyter notebook cell by cell. All the activity will have not been logged through LangSmith.